In [1]:
!pip install -q "unsloth[colab-new]" trl peft bitsandbytes accelerate
!pip install -q sentence-transformers nltk rouge-score pycocoevalcap optuna


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.1/81.1 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 8.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 42.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 84.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 48.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 66.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 49.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.4/24.4 MB 52.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 5.5 MB/s eta 0:00

In [2]:
import os
import json
import random

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from PIL import Image
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


Device: cuda


In [3]:
DATA_DIR = "/kaggle/input/datasets/ayaanmustafa/vrs-bench/vrsbench_data"
ANNOT_FILE = os.path.join(DATA_DIR, "VRSBench_train.json")  # update filename if different
IMG_DIR = os.path.join(DATA_DIR, "Images_train", "Images_train")

with open(ANNOT_FILE, "r") as f:
    raw = json.load(f)

caption_rows = []

for item in raw:
    conversations = item["conversations"]

    if "[caption]" in conversations[0]["value"].lower():
        caption_rows.append({
            "id": item["id"],
            "image": item["image"],
            "caption": conversations[1]["value"]
        })

df = pd.DataFrame(caption_rows)
# Normalize columns: expect a filename column and a caption column
# (adjust the renames below to match the printed column names above)
rename_map = {}
for c in df.columns:
    lc = c.lower()
    if "image" in lc and "path" not in lc:
        rename_map[c] = "filename"
    elif lc in ("image_path", "path"):
        rename_map[c] = "filename"
    elif "caption" in lc:
        rename_map[c] = "caption"

df = df.rename(columns=rename_map)
df["image_path"] = df["filename"].apply(lambda x: os.path.join(IMG_DIR, os.path.basename(str(x))))

df = df[["image_path", "caption"]].dropna().reset_index(drop=True)
print(df.shape)
df.head()


(20264, 2)


,image_path,caption
0,/kaggle/input/datasets/ayaanmustafa/vrs-bench/...,"The image, sourced from GoogleEarth, shows a r..."
1,/kaggle/input/datasets/ayaanmustafa/vrs-bench/...,This aerial image from Google Earth shows a de...
2,/kaggle/input/datasets/ayaanmustafa/vrs-bench/...,"The image, sourced from GoogleEarth, showcases..."
3,/kaggle/input/datasets/ayaanmustafa/vrs-bench/...,"The image, sourced from GoogleEarth, shows a f..."
4,/kaggle/input/datasets/ayaanmustafa/vrs-bench/...,"The image, sourced from GoogleEarth, showcases..."


## 2. Train / val split

In [14]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.1, random_state=SEED)

TRAIN_SUBSET = 5000    # cut for a ~1-1.5hr QLoRA run — VLM finetuning is far more expensive per-sample than BLIP
VAL_SUBSET = 50
IMG_SIZE = 256         # standard mid-size resolution; VRSBench source is 512x512 which blows up vision-token count on a VLM

if TRAIN_SUBSET:
    train_df = train_df.sample(n=min(TRAIN_SUBSET, len(train_df)), random_state=SEED).reset_index(drop=True)
if VAL_SUBSET:
    val_df = val_df.sample(n=min(VAL_SUBSET, len(val_df)), random_state=SEED).reset_index(drop=True)

print("Train:", len(train_df), "Val:", len(val_df))


Train: 5000 Val: 50


## 3. Load Qwen3-VL 4-bit + LoRA (Unsloth)
Rank 16 / alpha 16 — the report found this the best stability/overfitting trade-off vs rank 64.
LoRA targets vision encoder, language backbone, MLP, and attention modules (full-component adaptation,
not just the decoder — unlike the BLIP frozen-encoder approach).

In [5]:
from unsloth import FastVisionModel

MODEL_NAME = "unsloth/Qwen3-VL-4B-Instruct-bnb-4bit"

model, processor = FastVisionModel.from_pretrained(
    MODEL_NAME,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    random_state=SEED,
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.11: Fast Qwen3_Vl patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
total weights      : 3.190 GiB
no_split classes   : ['Qwen3VLTextDecoderLayer', 'Qwen3VLVisionBlock']
output head        : lm_head -> cuda:1
head headroom      : 0.359 GiB
activation reserve : 11.200 GiB requested
tied to head       : ['model.language_model.embed_tokens']
  cuda:0  budget  12.95 GiB  weights  1.714 GiB  free 11.238 GiB  reserve 11.200 GiB
  cuda:1  budget  13.00 GiB  weights  1.477 GiB  f

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


## 4. Prompt template — metric-aware, length-targeted
The report's key captioning lesson: zero/few-shot prompts produce overly long, redundant captions
that hurt BERT-BLEU4 via the length penalty. Use a system prompt that explicitly targets the
dataset's median caption length (~50 words) instead of a generic "describe this image" prompt.

In [15]:
SYSTEM_PROMPT = (
    "You are a remote-sensing image captioning assistant. "
    "Describe the satellite/aerial image in ONE concise paragraph of about 40-60 words. "
    "Mention key objects, their approximate spatial layout (e.g. top-left, bottom-right, center), "
    "and notable scene context. Do not pad with redundant detail or repeat phrases."
)

def build_conversation(image, caption=None):
    image = image.resize((IMG_SIZE, IMG_SIZE))

    messages = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": SYSTEM_PROMPT,
                }
            ],
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image,
                },
                {
                    "type": "text",
                    "text": "Caption this satellite image.",
                },
            ],
        },
    ]

    if caption is not None:
        messages.append(
            {
                "role": "assistant",
                "content": [
                    {
                        "type": "text",
                        "text": caption,
                    }
                ],
            }
        )

    return messages

## 5. PyTorch Dataset

In [16]:
from torch.utils.data import Dataset

class VRSBenchVLMDataset(Dataset):
    def __init__(self, df):
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        messages = build_conversation(image, row["caption"])
        return {"messages": messages}


train_dataset = VRSBenchVLMDataset(train_df)
val_dataset = VRSBenchVLMDataset(val_df)
print(len(train_dataset), len(val_dataset))


5000 50


## 6. Training — TRL SFTTrainer + Unsloth vision collator
Gradient accumulation for a larger effective batch on limited memory, matching the BLIP run's approach.
Mirrors the report's setup: 8-bit AdamW, cosine schedule with warmup, small per-device batch + accumulation.

In [8]:
from trl import SFTTrainer, SFTConfig
from unsloth.trainer import UnslothVisionDataCollator

FastVisionModel.for_training(model)

import torch

use_bf16 = torch.cuda.is_bf16_supported()

training_args = SFTConfig(
    output_dir="/kaggle/working/qwen3vl_vrsbench_lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,   # effective batch size 16
    num_train_epochs=1,              # 1 epoch over 3K samples — re-run with more epochs/data once timing is confirmed
    learning_rate=2e-4,
    optim="adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_steps=100,
    logging_steps=20,
    save_strategy="epoch",
    bf16=use_bf16,
    fp16=not use_bf16,
    remove_unused_columns=False,
    dataset_text_field="",
    dataset_kwargs={"skip_prepare_dataset": True},
    max_seq_length=384,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=processor,
    data_collator=UnslothVisionDataCollator(model, processor),
    train_dataset=train_dataset,
    args=training_args,
)

train_result = trainer.train()
print(train_result)


Unsloth: Model does not have a default image size - using 512


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 2
   \\   /|    Num examples = 5,000 | Num Epochs = 1 | Total steps = 313
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 39,321,600 of 4,477,137,408 (0.88% trained)
/usr/local/lib/python3.12/dist-packages/unsloth/import_fixes.py:4091: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  return original(name)
/usr/local/lib/python3.12/dist-packages/unsloth/import_fixes.py:4091: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  return original(name)
/usr/local/lib/python3.1

Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
20,3.174558
40,1.536801
60,0.836925
80,0.699044
100,0.659268
120,0.641117
140,0.634271
160,0.621957
180,0.609938
200,0.640146


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/qwen3vl_vrsbench_lora/checkpoint-313/tokenizer_config.json.


TrainOutput(global_step=313, training_loss=0.8593819514631083, metrics={'train_runtime': 4776.7373, 'train_samples_per_second': 1.047, 'train_steps_per_second': 0.066, 'total_flos': 5.257756628163379e+16, 'train_loss': 0.8593819514631083, 'epoch': 1.0})


In [9]:
from sentence_transformers import SentenceTransformer
import torch.nn.functional as F

bert_model = SentenceTransformer("all-MiniLM-L6-v2", device=str(DEVICE))

def get_ngrams(tokens, n):
    return [" ".join(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]

def bert_bleu4(candidate, reference, alpha=0.5):
    c_tokens = candidate.lower().split()
    r_tokens = reference.lower().split()

    c_ngrams_by_n = [get_ngrams(c_tokens, n) for n in range(1, 5)]
    r_ngrams_by_n = [get_ngrams(r_tokens, n) for n in range(1, 5)]

    all_texts = [g for grams in c_ngrams_by_n + r_ngrams_by_n for g in grams]
    if not all_texts:
        return 0.0

    embs = bert_model.encode(all_texts, convert_to_tensor=True, show_progress_bar=False, batch_size=256)
    embs = F.normalize(embs, dim=-1)

    idx = 0
    c_embs_by_n, r_embs_by_n = [], []
    for grams in c_ngrams_by_n:
        c_embs_by_n.append(embs[idx: idx + len(grams)])
        idx += len(grams)
    for grams in r_ngrams_by_n:
        r_embs_by_n.append(embs[idx: idx + len(grams)])
        idx += len(grams)

    log_p_sum = 0.0
    for n in range(4):
        C_emb, R_emb = c_embs_by_n[n], r_embs_by_n[n]
        if len(C_emb) == 0 or len(R_emb) == 0:
            p_n = 1e-9
        else:
            sims = torch.mm(R_emb, C_emb.T)
            p_n = sims.max(dim=1).values.mean().item()
            p_n = max(p_n, 1e-9)
        log_p_sum += np.log(p_n)

    L_c, L_r = len(c_tokens), max(len(r_tokens), 1)
    lp = np.exp(-alpha * abs(L_c - L_r) / L_r)
    return lp * np.exp(log_p_sum / 4)

def corpus_bert_bleu4(candidates, references):
    return float(np.mean([bert_bleu4(c, r) for c, r in zip(candidates, references)]))


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
nltk.download("punkt", quiet=True)

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from pycocoevalcap.cider.cider import Cider

smoothie = SmoothingFunction().method4
rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def compute_bleu_n(candidates, references, n):
    weights = tuple([1.0 / n] * n + [0.0] * (4 - n))
    scores = []
    for c, r in zip(candidates, references):
        c_tok, r_tok = c.lower().split(), [r.lower().split()]
        scores.append(sentence_bleu(r_tok, c_tok, weights=weights, smoothing_function=smoothie))
    return float(np.mean(scores))

def compute_meteor(candidates, references):
    return float(np.mean([meteor_score([r.lower().split()], c.lower().split()) for c, r in zip(candidates, references)]))

def compute_rouge_l(candidates, references):
    return float(np.mean([rouge.score(r, c)["rougeL"].fmeasure for c, r in zip(candidates, references)]))

def compute_cider(candidates, references):
    gts = {i: [r] for i, r in enumerate(references)}
    res = {i: [c] for i, c in enumerate(candidates)}
    score, _ = Cider().compute_score(gts, res)
    return float(score)

def compute_avg_len(candidates):
    return float(np.mean([len(c.split()) for c in candidates]))

def compute_all_metrics(candidates, references):
    return {
        "BLEU-1": compute_bleu_n(candidates, references, 1),
        "BLEU-2": compute_bleu_n(candidates, references, 2),
        "BLEU-3": compute_bleu_n(candidates, references, 3),
        "BLEU-4": compute_bleu_n(candidates, references, 4),
        "METEOR": compute_meteor(candidates, references),
        "ROUGE_L": compute_rouge_l(candidates, references),
        "CIDEr": compute_cider(candidates, references),
        "Avg_L": compute_avg_len(candidates),
        "BERT-BLEU4": corpus_bert_bleu4(candidates, references),
    }


## 8. Inference / validation loop

In [17]:
FastVisionModel.for_inference(model)

@torch.no_grad()
def generate_caption(image, max_new_tokens=110):
    messages = build_conversation(image, caption=None)
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to(DEVICE)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        use_cache=True,
        temperature=0.3,
        do_sample=False,
        repetition_penalty=1.3,
    )
    generated = output_ids[:, inputs["input_ids"].shape[1]:]
    text = processor.batch_decode(generated, skip_special_tokens=True)[0]
    return text.strip()


def validate(val_df, limit=None):
    subset = val_df if limit is None else val_df.iloc[:limit]
    preds, refs = [], []
    for _, row in tqdm(subset.iterrows(), total=len(subset), desc="val"):
        image = Image.open(row["image_path"]).convert("RGB")
        pred = generate_caption(image)
        preds.append(pred)
        refs.append(row["caption"])
    metrics = compute_all_metrics(preds, refs)
    return metrics, preds, refs


metrics, preds, refs = validate(val_df)
print(metrics)


val:   0%|          | 0/50 [00:00<?, ?it/s]

{'BLEU-1': 0.17205152850949798, 'BLEU-2': 0.07059492443868924, 'BLEU-3': 0.03385686241546297, 'BLEU-4': 0.019672376891962846, 'METEOR': 0.24158127289855152, 'ROUGE_L': 0.1669659471213443, 'CIDEr': 0.002485805068775941, 'Avg_L': 101.48, 'BERT-BLEU4': 0.40730434420978767}


## 9. Sample predictions

In [18]:
for p, r in list(zip(preds, refs))[:5]:
    print("PRED:", p)
    print("REF :", r)
    print("-" * 60)


PRED: The high-resolution GoogleEarth aerial view shows two large ships docked side by side on water bodies adjacent to land areas marked as 'land' at both ends within an enclosed space that appears like harbor facilities due to its structured nature around it. The left-most ship is larger than right one which has more visible details such as deck structures but smaller overall size compared to other object nearby.

There's no additional information provided regarding specific orientation angles for these vessels relative to each others nor any indication if they're moored securely against piers since there aren't clear
REF : The image shows two parallel ships on a body of water, captured via GoogleEarth. Both ships are of similar small sizes and are lying next to each other, extending from near the top of the image vertically down to the bottom. The ship on the left occupies the leftmost third of the image, while the ship on the right occupies the adjacent rightmost third, with no oth

## 10. Save LoRA adapter

In [19]:
model.save_pretrained("/kaggle/working/qwen3vl_vrsbench_lora_final")
processor.save_pretrained("/kaggle/working/qwen3vl_vrsbench_lora_final")
print("Saved.")


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/qwen3vl_vrsbench_lora_final/tokenizer_config.json.


Saved.
